# 安装 verl + vllm-ascend

本节将安装 RL 训练所需的三个核心组件：vLLM（推理引擎）、vLLM-Ascend（NPU 适配层）和 verl（RL 训练框架）。

安装完成后，你还将应用 Wordle AgentLoop 补丁，为后续训练做好准备。

---

## 1. 组件关系

Wordle RL 训练需要三个核心组件协同工作：

### 定位训练代码仓

下面的单元格会从当前课程仓自动查找同级的 `cann-recipes-train`，确认 Wordle recipe 已由 01.01 准备完成，然后把后续命令的工作目录切换到训练代码仓根目录。

In [ ]:
from pathlib import Path

current_dir = Path.cwd().resolve()
if (current_dir / 'llm_rl/qwen3_wordle').is_dir():
    recipe_root = current_dir
else:
    course_root = next(
        (path for path in (current_dir, *current_dir.parents)
         if (path / 'tutorials/rl_training_pipeline').is_dir()),
        None,
    )
    assert course_root is not None, '无法定位 cann-learning-hub 课程仓根目录'
    recipe_root = course_root.parent / 'cann-recipes-train'

assert (recipe_root / 'llm_rl/qwen3_wordle').is_dir(), (
    '未找到同级 cann-recipes-train，请先执行 01.01 中的仓库拉取单元格'
)
%cd {recipe_root}


下面的单元格会打印切换后的目录，并再次检查 `llm_rl/qwen3_wordle/` 是否存在；看到“目录检查通过”后再继续安装依赖。

In [ ]:
import os

# CANNLab 从 cann-learning-hub 课程仓进入；上一单元格已切换到同级训练代码仓
print(f'当前工作目录: {os.getcwd()}')
assert os.path.isdir('llm_rl/qwen3_wordle'), '未正确定位 cann-recipes-train 根目录'
print('目录检查通过: llm_rl/qwen3_wordle/ 存在')


<img src="./images/stack_layers.png" alt="Wordle RL 环境栈" width="90%">

这三个组件是分层关系：外层的 verl 负责调度 RL 训练流程，中层的 vLLM / vllm-ascend 负责在 rollout 阶段生成模型回复，底层的 Ascend NPU 和 CANN 提供实际计算能力。

**verl** 负责管理训练循环：收集 rollout → 计算 reward → 计算 advantage → 更新策略。

**vLLM** 负责生成 rollout：在训练过程中让模型生成文本（如 Wordle 猜词），以 OpenAI API 兼容接口提供服务。

**vLLM-Ascend** 是 vLLM 的 NPU 适配层，让 vLLM 能在 Ascend NPU 上运行。

---

## 2. 版本配套表

| 组件 | 版本 | 说明 |
|------|------|------|
| CANN | 9.0.0 | NPU 驱动 + 工具链 |
| torch | 2.9.0 | PyTorch |
| torch_npu | 2.9.0 | PyTorch NPU 适配 |
| vllm | 0.18.0 | 推理引擎 |
| vllm-ascend | 0.18.0 | vLLM NPU 适配 |
| verl | 0.8.0 | RL 训练框架 |

---

## 3. 安装步骤

本课程从 `cann-learning-hub` 课程仓进入 CANNLab，并在 01.01 中将训练代码仓克隆到课程仓同级目录，目录结构如下：

```text
/mnt/workspace/gitCode/cann/
├── cann-learning-hub/         <- CANNLab 入口及课程 notebook
└── cann-recipes-train/        <- 01.01 拉取的训练代码仓
    └── llm_rl/qwen3_wordle/   <- 训练工作目录
```

下方命令中：vLLM / vLLM-Ascend 安装到 `/home/developer/`（避免与训练目录产生 Python 导入冲突）；verl 和训练依赖安装到 `llm_rl/qwen3_wordle/` 下。

### 3.1 安装 vLLM

In [ ]:
%%bash
# vLLM 安装到 /home/developer/ 下，避免与训练工作目录产生 Python 导入冲突
cd /home/developer
if [ ! -d vllm/.git ]; then
  git clone --depth 1 --branch v0.18.0 https://github.com/vllm-project/vllm
else
  echo '复用已存在的 /home/developer/vllm'
fi
cd vllm && VLLM_TARGET_DEVICE=empty pip install -v -e .


### 3.2 安装 vLLM-Ascend

vLLM-Ascend 需要在当前 CANN 环境中从源码编译。下面的单元格先检查 GCC/G++ 版本；低于 11 时自动升级到 11，已经满足 11+ 时不会改动编译器。

In [ ]:
%%bash
set -euo pipefail

# vLLM-Ascend 源码编译要求 GCC/G++ 11 或更高版本
GCC_VERSION="$(gcc -dumpfullversion -dumpversion 2>/dev/null || true)"
GCC_MAJOR="${GCC_VERSION%%.*}"
if ! [[ "$GCC_MAJOR" =~ ^[0-9]+$ ]]; then
  GCC_MAJOR=0
fi
echo "当前 GCC 版本: ${GCC_VERSION:-未安装}"

if (( GCC_MAJOR < 11 )); then
  echo 'GCC 版本低于 11，开始安装 GCC/G++ 11'
  SUDO=()
  if (( EUID != 0 )); then
    command -v sudo >/dev/null || { echo '当前用户不是 root，且系统未安装 sudo' >&2; exit 1; }
    sudo -n true || { echo '升级 GCC 需要免密 sudo 权限' >&2; exit 1; }
    SUDO=(sudo)
  fi
  "${SUDO[@]}" apt-get update
  "${SUDO[@]}" env DEBIAN_FRONTEND=noninteractive apt-get install -y software-properties-common
  "${SUDO[@]}" add-apt-repository -y ppa:ubuntu-toolchain-r/test
  "${SUDO[@]}" apt-get update
  "${SUDO[@]}" env DEBIAN_FRONTEND=noninteractive apt-get install -y gcc-11 g++-11
  "${SUDO[@]}" update-alternatives --install /usr/bin/gcc gcc /usr/bin/gcc-11 110
  "${SUDO[@]}" update-alternatives --install /usr/bin/g++ g++ /usr/bin/g++-11 110
  "${SUDO[@]}" update-alternatives --set gcc /usr/bin/gcc-11
  "${SUDO[@]}" update-alternatives --set g++ /usr/bin/g++-11
  hash -r
else
  echo 'GCC 已满足 11+，跳过升级'
fi

GCC_VERSION="$(gcc -dumpfullversion -dumpversion)"
GXX_VERSION="$(g++ -dumpfullversion -dumpversion)"
GCC_MAJOR="${GCC_VERSION%%.*}"
if (( GCC_MAJOR < 11 )); then
  echo "GCC 升级失败，当前版本: $GCC_VERSION" >&2
  exit 1
fi
echo "GCC/G++ 检查通过: gcc=$GCC_VERSION, g++=$GXX_VERSION"


编译器检查通过后，下面的单元格会下载或复用 vLLM-Ascend v0.18.0、初始化其子模块，并以 editable 模式完成源码编译安装。由于每个 `%%bash` 都是独立子进程，单元格会在编译前加载并校验 CANN 环境，避免 `ASCEND_OPP_PATH` 未设置导致 `torch_npu` 加载失败。

In [ ]:
%%bash
set -euo pipefail

CANN_HOME="${ASCEND_HOME_PATH:-/home/developer/Ascend/cann}"
CANN_ENV="${CANN_HOME}/set_env.sh"
if [ ! -f "${CANN_ENV}" ]; then
  echo "未找到 CANN 环境脚本: ${CANN_ENV}" >&2
  exit 1
fi
source "${CANN_ENV}"
: "${ASCEND_OPP_PATH:?加载 CANN 环境后 ASCEND_OPP_PATH 仍未设置}"
test -d "${ASCEND_OPP_PATH}"
echo "CANN 环境检查通过: ASCEND_OPP_PATH=${ASCEND_OPP_PATH}"

cd /home/developer
if [ ! -d vllm-ascend/.git ]; then
  git clone --depth 1 --branch v0.18.0 https://github.com/vllm-project/vllm-ascend.git
else
  echo '复用已存在的 /home/developer/vllm-ascend'
fi
cd vllm-ascend
git submodule update --init --recursive
C_COMPILER="$(command -v gcc)" CXX_COMPILER="$(command -v g++)" python3 -m pip install -v -e .


### 3.3 安装 verl

In [ ]:
%%bash
cd llm_rl/qwen3_wordle
if [ ! -d verl/.git ]; then
  git clone --depth 1 --branch v0.8.0 https://github.com/verl-project/verl
else
  echo '复用已存在的 verl'
fi
cd verl && pip install -v -e .


### 3.4 安装 Wordle 训练依赖

verl 会自动安装大部分依赖（pyarrow、pandas、ray、wandb 等），但 Wordle 训练还需要 nltk 和 textarena：

In [ ]:
%%bash
cd llm_rl/qwen3_wordle
pip install -r requirements.txt

# NLTK 数据（TextArena Wordle 依赖 pos_tag 过滤词性）
python3 -c "import nltk; nltk.download('words'); nltk.download('averaged_perceptron_tagger_eng')"


### 3.5 应用 Wordle AgentLoop 补丁

Wordle 训练需要一个自定义的 `WordleAgentLoop` 组件。通过 patch 将其注册到 verl 源码中：

In [ ]:
%%bash
cd llm_rl/qwen3_wordle/verl
PATCH=../patches/0001-wordle-agent-loop.patch
if git apply --reverse --check "$PATCH" >/dev/null 2>&1; then
  echo 'Wordle AgentLoop 补丁已经应用，跳过'
elif git apply --check "$PATCH"; then
  git apply "$PATCH"
  echo 'Wordle AgentLoop 补丁应用成功'
else
  echo '补丁与当前 verl 工作区冲突，请确认使用干净的 v0.8.0' >&2
  exit 1
fi


---

## 4. 验证安装

运行以下命令，确认所有组件导入正常：

In [ ]:
%%bash
set -euo pipefail

CANN_HOME="${ASCEND_HOME_PATH:-/home/developer/Ascend/cann}"
source "${CANN_HOME}/set_env.sh"
: "${ASCEND_OPP_PATH:?加载 CANN 环境后 ASCEND_OPP_PATH 仍未设置}"
CUSTOM_OP_ENV=/home/developer/vllm-ascend/vllm_ascend/_cann_ops_custom/vendors/vllm-ascend/bin/set_env.bash
if [ -f "${CUSTOM_OP_ENV}" ]; then
  export ASCEND_CUSTOM_OPP_PATH="${ASCEND_CUSTOM_OPP_PATH:-}"
  export LD_LIBRARY_PATH="${LD_LIBRARY_PATH:-}"
  source "${CUSTOM_OP_ENV}"
fi

cd llm_rl/qwen3_wordle
python3 -c "
import torch
import torch_npu
import vllm
import vllm_ascend
from verl.experimental.agent_loop.wordle_agent_loop import WordleAgentLoop
from importlib.metadata import version
print('torch:', version('torch'))
print('torch-npu:', version('torch-npu'))
print('verl:', version('verl'))
print('vllm:', version('vllm'))
print('vllm-ascend:', version('vllm-ascend'))
print('All imports OK')
"


---

## 5. 常见问题

**Q: `No module named 'verl'`？**
A: 确认在 verl 目录下执行了 `pip install -v -e .`。

**Q: AgentLoop 补丁失败？**
A: 确认 verl 在 v0.8.0 版本（`git describe --tags`），补丁是针对此版本生成的。

**Q: NLTK 下载失败（SSL 错误）？**
A: 网络不通时需配置代理后重试。

---

## 课后练习

1. （判断题）vLLM 在 RL 训练中的主要作用是生成 rollout（模型回复），供奖励函数打分。

2. （判断题）verl 的 `pip install -e .` 会自动安装 pyarrow、pandas、ray 等依赖，无需手动安装。

3. （判断题）Wordle AgentLoop 补丁只需要复制 `wordle_agent_loop.py` 文件即可，不需要修改 verl 的 `__init__.py`。

4. （单选题）以下哪个组件负责在 RL 训练中管理推理、奖励、训练的循环？
    A. vLLM
    B. vLLM-Ascend
    C. verl
    D. CANN

5. （单选题）安装 verl 后还需要单独安装以下哪个依赖？
    A. pyarrow
    B. ray
    C. nltk
    D. pandas

> 完成上面的练习后，再运行下方单元格查看参考答案和解析。

In [ ]:
from pathlib import Path
import subprocess

repo_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
course_root = repo_root if (repo_root / 'tutorials/rl_training_pipeline').is_dir() else repo_root.parent / 'cann-learning-hub'
answer_path = course_root / 'tutorials/rl_training_pipeline/01_environment_setup/answer/01.02_answer.txt'
assert answer_path.is_file(), f'未找到答案文件: {answer_path}'
print(answer_path.read_text(encoding='utf-8'))